# 面试问题：关键词、向量、实体等多路召回的配额、融合和超时降级怎样设计？

**一句话回答**：先按 query family 定义候选覆盖目标和总延迟/候选预算；各通道独立返回带 provenance 的稳定列表，coordinator 做 ACL、去重、配额和融合；离线用 marginal recall/成本选择预算，线上每路有 deadline/circuit breaker，超时后从已完成通道降级，不能让最慢一路拖垮整体。

本 Notebook 用小型受控结果集实现 quota、RRF、边际预算搜索、全局去重、timeout fallback、trace 与 query-level bootstrap。

In [ ]:
from dataclasses import dataclass
from collections import Counter, defaultdict
import hashlib, json, math
import numpy as np

CHANNELS83=("keyword","dense","entity","fresh")
assert len(CHANNELS83)==4 and len(set(CHANNELS83))==4
assert np.isfinite([1,2,3]).all()
assert hashlib.sha256(b"keyword").hexdigest()!=hashlib.sha256(b"dense").hexdigest()

## 1. 通道返回合同

每个 hit 包含 query、doc、channel、rank、raw score、tenant、index version 和 latency。raw score 跨通道不可直接相加：BM25、cosine、图遍历分数尺度不同。RRF 只使用 rank，是稳健 baseline；若学权重，需要独立训练数据和校准。

ACL 应在通道内部尽早过滤，coordinator 再做 defense-in-depth；所有列表必须稳定排序，便于重放。

In [ ]:
@dataclass(frozen=True)
class Hit83:
    query_id:str; doc_id:str; channel:str; rank:int; score:float; tenant:str; version:str
    def __post_init__(self):
        if self.channel not in CHANNELS83 or self.rank<1 or not math.isfinite(self.score) or not self.doc_id: raise ValueError("hit_contract")
h83=Hit83("q0","d1","keyword",1,3.2,"t1","idx-1")
assert h83.rank==1 and h83.channel=="keyword"
try: Hit83("q","d","bad",0,float("nan"),"t","v"); raise AssertionError("bad hit accepted")
except ValueError as e: assert str(e)=="hit_contract"
assert set(CHANNELS83)=={"keyword","dense","entity","fresh"}

## 2. 构造互补通道与 query family

关键词擅长精确术语，dense 擅长改写，entity 擅长实体关系，fresh 擅长新内容。如果所有通道排序完全相同，多路召回只是增加成本。受控数据让不同 family 的 gold 出现在不同通道，用于验证配额选择。

线上 gold 来自点击、人工标注或任务成功，但必须处理曝光偏差；本例直接给出 oracle，只验证 coordinator。

In [ ]:
families83=["exact","semantic","entity","fresh"]
queries83=[]; results83={}; gold83={}
for qi in range(80):
    q=f"q{qi:02d}"; family=families83[qi%4]; queries83.append((q,family)); gold={f"{q}-g0",f"{q}-g1"}; gold83[q]=gold
    for channel in CHANNELS83:
        favored=(channel=={"exact":"keyword","semantic":"dense","entity":"entity","fresh":"fresh"}[family])
        docs=([f"{q}-g0",f"{q}-g1"] if favored else [f"{q}-{channel}-x0",f"{q}-{channel}-x1"])+[f"{q}-common",f"{q}-{channel}-x2",f"{q}-{channel}-x3"]
        results83[(q,channel)]=[Hit83(q,d,channel,r+1,1/(r+1),"t1","idx-1") for r,d in enumerate(docs)]
assert len(queries83)==80 and len(results83)==320
assert all(len(v)==5 for v in results83.values())
assert all(len(g)==2 for g in gold83.values())

## 3. 配额截断、去重与 RRF

每通道先取 quota，再按 doc ID 聚合。RRF 得分 `Σ 1/(k+rank)`；同一文档被多路召回会得到更高分，但最终只出现一次并保留所有 provenance。`k` 越大，头部 rank 差异越弱。

tie-break 使用 doc ID，避免字典遍历顺序影响线上结果。

In [ ]:
def fuse83(query_id,quotas,rrf_k=60,completed=CHANNELS83):
    if set(quotas)!=set(CHANNELS83) or any(not isinstance(v,int) or v<0 for v in quotas.values()): raise ValueError("quota_contract")
    agg=defaultdict(lambda:{"rrf":0.,"channels":[],"ranks":{}})
    for channel in completed:
        for h in results83[(query_id,channel)][:quotas[channel]]:
            if h.tenant!="t1": continue
            agg[h.doc_id]["rrf"]+=1/(rrf_k+h.rank); agg[h.doc_id]["channels"].append(channel); agg[h.doc_id]["ranks"][channel]=h.rank
    return sorted([(v["rrf"],doc,tuple(sorted(v["channels"])),v["ranks"]) for doc,v in agg.items()],key=lambda x:(-x[0],x[1]))
equal83={c:3 for c in CHANNELS83}; fused83=fuse83("q00",equal83)
assert len({x[1] for x in fused83})==len(fused83)
assert next(x for x in fused83 if x[1]=="q00-common")[2]==tuple(sorted(CHANNELS83))
assert fused83==fuse83("q00",equal83)

## 4. 用通道消融验证“互补”而不是看列表数量

通道贡献应以移除该路后的任务指标下降衡量，而不是它返回了多少候选。大量重复候选可能贡献为零；某个低流量 entity 通道却可能决定实体 query 是否有任何 gold。消融需要按 query family 报告。

下面在每个 family 上移除对应优势通道，Recall 应下降；这也是 circuit breaker 降级预算的依据。

In [ ]:
favored83={"exact":"keyword","semantic":"dense","entity":"entity","fresh":"fresh"}
def recall_from83(q,quotas,completed):
    docs=[x[1] for x in fuse83(q,quotas,completed=completed)[:10]]
    return len(set(docs)&gold83[q])/len(gold83[q])
for fam in families83:
    q=next(q for q,f in queries83 if f==fam); full_recall=recall_from83(q,equal83,CHANNELS83); completed=tuple(c for c in CHANNELS83 if c!=favored83[fam]); ablated=recall_from83(q,equal83,completed)
    assert full_recall==1 and ablated<full_recall
assert len(favored83)==4
assert set(favored83.values())==set(CHANNELS83)

## 5. 用边际 Recall/成本分配离线预算

从零 quota 开始，每次尝试给某通道多一个候选，计算验证集 recall 增量除以 latency/candidate cost，选择收益最大的动作，直到候选预算耗尽。它是可解释 greedy baseline，不保证全局最优，但比拍脑袋固定 `top_k=50` 更容易审计。

预算应按 query family 分开学习；全局平均会牺牲少数但重要的实体/新鲜查询。

In [ ]:
cost83={"keyword":1.0,"dense":2.2,"entity":1.6,"fresh":1.3}
def recall83(q,quotas,completed=CHANNELS83,k=10):
    docs=[x[1] for x in fuse83(q,quotas,completed=completed)[:k]]; return len(set(docs)&gold83[q])/len(gold83[q])
def learn_quota83(train_queries,total_candidates=8):
    quota={c:0 for c in CHANNELS83}
    while sum(quota.values())<total_candidates:
        base=np.mean([recall83(q,quota) for q in train_queries]); choices=[]
        for c in CHANNELS83:
            trial=dict(quota); trial[c]+=1; gain=np.mean([recall83(q,trial) for q in train_queries])-base; choices.append((gain/cost83[c],gain,-cost83[c],c))
        _,_,_,best=max(choices); quota[best]+=1
    return quota
family_quota83={f:learn_quota83([q for q,fam in queries83[:60] if fam==f],8) for f in families83}
assert all(sum(q.values())==8 for q in family_quota83.values())
assert family_quota83["exact"]["keyword"]>=max(family_quota83["exact"][c] for c in CHANNELS83 if c!="keyword")
assert family_quota83["semantic"]["dense"]>=2 and family_quota83["entity"]["entity"]>=2

## 6. 与均匀配额比较，并按 family 切片

评估单位是 query，不是 hit。比较 learned family quota 与每路两个候选的 uniform baseline；在这个 oracle 数据里，两者总候选数相同，learned 应保持或提高 Recall@10。还要报告 per-family，防止平均数掩盖某一路退化。

若 reranker 只能处理固定候选数，fusion 输出还要严格截断并记录各通道最终贡献。

In [ ]:
uniform83={c:2 for c in CHANNELS83}; test83=queries83[60:]
learned_scores83=[]; uniform_scores83=[]; slices83=defaultdict(list)
for q,fam in test83:
    lr=recall83(q,family_quota83[fam]); ur=recall83(q,uniform83); learned_scores83.append(lr); uniform_scores83.append(ur); slices83[fam].append(lr)
assert np.mean(learned_scores83)>=np.mean(uniform_scores83)
assert all(np.mean(v)==1 for v in slices83.values())
assert all(0<=x<=1 for x in learned_scores83) and len(learned_scores83)==20

## 7. 每路 deadline、降级和 circuit breaker

coordinator 不等待最慢通道：在总 deadline 前只合并已完成列表，并在 trace 中标出 timed out channel。若 dense 连续超时，circuit breaker 暂停调用并把预算转给 keyword/entity；恢复时先小流量 probe。

降级不是返回空结果。RRF 可对任意完成子集工作，但质量报告必须按缺失通道切片。

In [ ]:
latency83={"keyword":4,"dense":24,"entity":9,"fresh":7}
def serve83(q,fam,deadline_ms):
    completed=tuple(c for c in CHANNELS83 if latency83[c]<=deadline_ms); quota=family_quota83[fam]
    ranked=fuse83(q,quota,completed=completed)[:10]
    trace={"query":q,"completed":completed,"timed_out":tuple(c for c in CHANNELS83 if c not in completed),"candidates":len(ranked)}
    return ranked,trace
degraded83,trace83=serve83("q61","semantic",10)
assert "dense" in trace83["timed_out"] and set(trace83["completed"])=={"keyword","entity","fresh"}
assert degraded83 and len({x[1] for x in degraded83})==len(degraded83)
full83,full_trace83=serve83("q61","semantic",30)
assert len(full_trace83["timed_out"])==0 and recall83("q61",family_quota83["semantic"])==1

## 8. Query-level bootstrap 与发布合同

候选 hit 同属一个 query，不能当独立样本做置信区间。下面按 query 重采样 Recall 差值。manifest 绑定通道版本、family classifier、quota、RRF k、deadline 与 ACL schema；更新任一通道都要重放固定 query set。

线上观察：每路成功/超时率、唯一候选贡献、重复率、最终 top-k 占比、per-family 任务指标和总尾延迟。

In [ ]:
diffs83=np.array(learned_scores83)-np.array(uniform_scores83); boot_rng83=np.random.default_rng(8301)
boots83=np.array([diffs83[boot_rng83.integers(0,len(diffs83),len(diffs83))].mean() for _ in range(500)])
ci83=np.quantile(boots83,[.025,.975])
manifest83={"schema":1,"channels":list(CHANNELS83),"versions":{c:"idx-1" for c in CHANNELS83},"quota":family_quota83,"rrf_k":60,"deadline_ms":30}
digest83=hashlib.sha256(json.dumps(manifest83,sort_keys=True,separators=(",",":")).encode()).hexdigest()
assert ci83[0]<=diffs83.mean()<=ci83[1] and len(digest83)==64
assert set(manifest83["quota"])==set(families83) and manifest83["deadline_ms"]>max(latency83.values())
assert all(set(q)==set(CHANNELS83) for q in manifest83["quota"].values())
print({"uniform":np.mean(uniform_scores83),"learned":np.mean(learned_scores83),"ci":ci83.tolist()})

## 9. 面试收束、参考与练习

回答闭环：通道合同与互补性 → query family/总预算 → quota + 去重 + RRF → marginal recall/cost → per-query 评估 → deadline/circuit breaker → trace/版本/回滚。不要把“BM25 和向量各取 50 再 RRF”当成完整设计。

练习：加入 learned fusion；让 quota optimizer 同时受 latency 约束；模拟某通道结果版本落后；实现 ACL 过滤前后候选计数但不泄露文档 ID。

参考：[Reciprocal Rank Fusion 原论文](https://dl.acm.org/doi/10.1145/1571941.1572114)、[nDCG 原始讨论](https://dl.acm.org/doi/10.1145/582415.582418)、[RAG 原论文](https://arxiv.org/abs/2005.11401)。